# Phase 5 -- SQL: PostgreSQL Queries for HR Attrition Analysis

**Goal:** Answer every Phase 1-3 business question using SQL.

## Why SQL matters

SQL is the most-tested technical skill in Indian DA/BA interviews. Most job listings require it.

## Setup

```bash
pip install psycopg2-binary sqlalchemy pandas
```

Change `DB_PASSWORD` below. Falls back to SQLite automatically if PostgreSQL is not installed.

## 20 Queries covering:

SELECT/WHERE/ORDER BY, GROUP BY + aggregates, HAVING, CASE WHEN, Subqueries, Window functions (RANK/ROW_NUMBER), CTEs, CREATE TABLE (star schema), Views


## Step 1 -- Connect and Load Data

**What this cell does:** Connects to PostgreSQL (or SQLite fallback), writes the cleaned DataFrame as a SQL table, defines a helper `sql()` function.


In [6]:
import pandas as pd
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/ibm_hr_stats_ready.csv')
print('Loaded: ' + str(df.shape[0]) + ' rows x ' + str(df.shape[1]) + ' columns')

DB_USER = 'postgres'
DB_PASSWORD = 'admin'
DB_HOST = 'localhost'
DB_PORT = '5432'
DB_NAME = 'ibm_hr_db'

try:
    engine = create_engine(
        'postgresql://' + DB_USER + ':' + DB_PASSWORD + '@' + DB_HOST + ':' + DB_PORT + '/' + DB_NAME
    )
    with engine.connect() as conn:
        conn.execute(text('SELECT 1'))
    df.to_sql('employees', engine, if_exists='replace', index=False)
    print('Connected: PostgreSQL')
    print('Table employees created: ' + str(len(df)) + ' rows')
except Exception:
    import sqlite3
    sqlite3.connect('../data/ibm_hr.db').close()
    engine = create_engine('sqlite:///../data/ibm_hr.db')
    df.to_sql('employees', engine, if_exists='replace', index=False)
    print('Using SQLite fallback -- all queries work identically')

def sql(query, label=''):
    result = pd.read_sql(query, engine)

    if label:
        print('\n=== ' + label + ' ===')
    print(result.to_string(index=False))
    return result

Loaded: 1470 rows x 39 columns
Using SQLite fallback -- all queries work identically




**Result:** Connected. The `employees` table has 1470 rows -- one per employee, queryable with SQL.


In [3]:
# Step 1.5 -- Show Employees table column names one by one
print("=== Employees Table Columns (39 total) ===")
for i, col_name in enumerate(df.columns, 1):
    print(f"{i:2d}. {col_name}")
print(f"\nTotal columns: {len(df.columns)}")


=== Employees Table Columns (39 total) ===
 1. Age
 2. Attrition
 3. BusinessTravel
 4. DailyRate
 5. Department
 6. DistanceFromHome
 7. Education
 8. EducationField
 9. EmployeeNumber
10. EnvironmentSatisfaction
11. Gender
12. HourlyRate
13. JobInvolvement
14. JobLevel
15. JobRole
16. JobSatisfaction
17. MaritalStatus
18. MonthlyIncome
19. MonthlyRate
20. NumCompaniesWorked
21. OverTime
22. PercentSalaryHike
23. PerformanceRating
24. RelationshipSatisfaction
25. StockOptionLevel
26. TotalWorkingYears
27. TrainingTimesLastYear
28. WorkLifeBalance
29. YearsAtCompany
30. YearsInCurrentRole
31. YearsSinceLastPromotion
32. YearsWithCurrManager
33. Income_Zscore
34. IsHighRisk
35. AnnualIncome
36. SatisfactionScore
37. TenureGroup
38. DeptAvgIncome
39. Below_Dept_Avg

Total columns: 39


In [2]:
df.head(5)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,...,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Income_Zscore,IsHighRisk,AnnualIncome,SatisfactionScore,TenureGroup,DeptAvgIncome,Below_Dept_Avg
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,...,4,0,5,-0.108,0,71916,2.00,6-10,6959.172646,1
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,...,7,1,7,-0.292,0,61560,3.00,6-10,6281.252862,1
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,...,0,0,0,-0.938,0,25080,3.00,0-2,6281.252862,1
3,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,5,4,...,7,3,0,-0.764,0,34908,3.25,6-10,6281.252862,1
4,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,7,1,...,2,2,2,-0.645,0,41616,2.50,0-2,6281.252862,1


In [3]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,1470.0,36.923810,9.135373,18.000000,30.000000,36.000000,43.000000,60.000000
Attrition,1470.0,0.161224,0.367863,0.000000,0.000000,0.000000,0.000000,1.000000
DailyRate,1470.0,802.485714,403.509100,102.000000,465.000000,802.000000,1157.000000,1499.000000
DistanceFromHome,1470.0,9.192517,8.106864,1.000000,2.000000,7.000000,14.000000,29.000000
Education,1470.0,2.912925,1.024165,1.000000,2.000000,3.000000,4.000000,5.000000
EmployeeNumber,1470.0,1024.865306,602.024335,1.000000,491.250000,1020.500000,1555.750000,2068.000000
EnvironmentSatisfaction,1470.0,2.721769,1.093082,1.000000,2.000000,3.000000,4.000000,4.000000
Gender,1470.0,0.600000,0.490065,0.000000,0.000000,1.000000,1.000000,1.000000
HourlyRate,1470.0,65.891156,20.329428,30.000000,48.000000,66.000000,83.750000,100.000000
JobInvolvement,1470.0,2.729932,0.711561,1.000000,2.000000,3.000000,3.000000,4.000000


## Step 2 -- Basic Queries: SELECT, WHERE, ORDER BY

**The foundation of all SQL.** These 4 queries verify your Phase 1 baseline numbers.


In [4]:
## 1. Finding total number of employees
sql('SELECT COUNT(*) AS Total_employees FROM employees','Total Number of Employees')


=== Total Number of Employees ===
 total_employees
            1470


,total_employees
0,1470


In [14]:
### === Q2: Overall attrition rate ===
sql('SELECT COUNT(*) AS total, ' \
'SUM(CAST("Attrition" AS INTEGER)) AS left_company, ' \
'ROUND((AVG(CAST("Attrition" AS FLOAT)) * 100.0)::numeric,2) AS attrition_pct FROM employees', 
'Q2: Overall attrition rate')



=== Q2: Overall attrition rate ===
 total  left_company  attrition_pct
  1470           237          16.12


,total,left_company,attrition_pct
0,1470,237,16.12


In [ ]:
###  === Q3: Top 10 highest-paid leavers ===
sql('SELECT "JobRole", "MonthlyIncome", "Age", "YearsAtCompany" ' \
'FROM employees ' \
'WHERE "Attrition" = 1 ORDER BY "MonthlyIncome" DESC LIMIT 10', 
'Q3: Top 10 highest-paid leavers')



=== Q3: Top 10 highest-paid leavers ===
          JobRole  MonthlyIncome  Age  YearsAtCompany
          Manager          19859   55               5
          Manager          19845   52              32
Research Director          19545   41              22
Research Director          19246   58              31
          Manager          18824   45              24
  Sales Executive          13758   42              21
  Sales Executive          13695   55              19
          Manager          13610   33               7
  Sales Executive          13194   40               1
  Sales Executive          12936   47              23


,JobRole,MonthlyIncome,Age,YearsAtCompany
0,Manager,19859,55,5
1,Manager,19845,52,32
2,Research Director,19545,41,22
3,Research Director,19246,58,31
4,Manager,18824,45,24
5,Sales Executive,13758,42,21
6,Sales Executive,13695,55,19
7,Manager,13610,33,7
8,Sales Executive,13194,40,1
9,Sales Executive,12936,47,23


In [38]:
### === Q4: Overtime + Low Income danger zone ===
sql('''
        SELECT COUNT(*) AS employees_in_segment, 
        ROUND((AVG(CAST("Attrition" AS FLOAT)) * 100.0)::numeric,1) AS attrition_pct,
        SUM(CAST("Attrition" AS INTEGER)) AS left_company
        FROM employees WHERE "OverTime" = 1 AND "MonthlyIncome" < 5000
    ''', 'Q4: Overtime + Low Income danger zone')


=== Q4: Overtime + Low Income danger zone ===
 employees_in_segment  attrition_pct  left_company
                  205           42.9            88


,employees_in_segment,attrition_pct,left_company
0,205,42.9,88


**Q2 confirms:** 16.12% attrition -- matches Phase 1 exactly.
**Q4 confirms:** Phase 1 Finding 3 -- the danger zone.

**Keywords:** SELECT (columns), FROM (table), WHERE (filter rows), ORDER BY (sort), LIMIT (top N), COUNT/SUM/AVG (aggregates).


## Step 3 -- GROUP BY + Aggregates (Most Important SQL Pattern)

**The most-asked SQL pattern in DA interviews.** These 4 queries confirm all Phase 1 categorical findings with SQL.


In [43]:
#### 1. Attrition by department
sql('''
    SELECT "Department", COUNT(*) AS total,
           SUM(CAST("Attrition" AS INTEGER)) AS left_company,
           ROUND((AVG(CAST("Attrition" AS FLOAT)) * 100.0)::Numeric, 2) AS attrition_pct,
           ROUND((AVG("MonthlyIncome"))::Numeric, 2) AS avg_income
    FROM employees GROUP BY "Department" 
    ORDER BY attrition_pct DESC
''', 'Q5: Attrition by Department')


=== Q5: Attrition by Department ===
            Department  total  left_company  attrition_pct  avg_income
                 Sales    446            92          20.63     6959.17
       Human Resources     63            12          19.05     6654.51
Research & Development    961           133          13.84     6281.25


,Department,total,left_company,attrition_pct,avg_income
0,Sales,446,92,20.63,6959.17
1,Human Resources,63,12,19.05,6654.51
2,Research & Development,961,133,13.84,6281.25


In [45]:
#### 2. Attrition by JobRole

sql('''
    SELECT "JobRole", COUNT(*) AS total,
           ROUND((AVG(CAST("Attrition" AS FLOAT)) * 100.0)::Numeric, 2) AS attrition_pct,
           ROUND(AVG("MonthlyIncome")::Numeric, 2) AS avg_income
    FROM employees GROUP BY "JobRole" ORDER BY attrition_pct DESC
''', 'Q6: Attrition by JobRole')


=== Q6: Attrition by JobRole ===
                  JobRole  total  attrition_pct  avg_income
     Sales Representative     83          39.76     2626.00
    Laboratory Technician    259          23.94     3237.17
          Human Resources     52          23.08     4235.75
          Sales Executive    326          17.48     6924.28
       Research Scientist    292          16.10     3239.97
   Manufacturing Director    145           6.90     7295.14
Healthcare Representative    131           6.87     7528.76
                  Manager    102           4.90    17181.68
        Research Director     80           2.50    16033.55


,JobRole,total,attrition_pct,avg_income
0,Sales Representative,83,39.76,2626.00
1,Laboratory Technician,259,23.94,3237.17
2,Human Resources,52,23.08,4235.75
3,Sales Executive,326,17.48,6924.28
4,Research Scientist,292,16.10,3239.97
5,Manufacturing Director,145,6.90,7295.14
6,Healthcare Representative,131,6.87,7528.76
7,Manager,102,4.90,17181.68
8,Research Director,80,2.50,16033.55


In [46]:
#### 3. Attrition by MaritalStatus
sql('''
    SELECT "MaritalStatus", COUNT(*) AS total,
           ROUND((AVG(CAST("Attrition" AS FLOAT)) * 100.0)::Numeric, 1) AS attrition_pct
    FROM employees GROUP BY "MaritalStatus" ORDER BY attrition_pct DESC
''', 'Q7: Attrition by MaritalStatus')


=== Q7: Attrition by MaritalStatus ===
MaritalStatus  total  attrition_pct
       Single    470           25.5
      Married    673           12.5
     Divorced    327           10.1


,MaritalStatus,total,attrition_pct
0,Single,470,25.5
1,Married,673,12.5
2,Divorced,327,10.1


In [47]:
#### 4. Attrition by BusinessTravel
sql('''
    SELECT "BusinessTravel", COUNT(*) AS total,
           ROUND((AVG(CAST("Attrition" AS FLOAT)) * 100.0)::Numeric, 1) AS attrition_pct
    FROM employees GROUP BY "BusinessTravel" ORDER BY attrition_pct DESC
''', 'Q8: Attrition by BusinessTravel')


=== Q8: Attrition by BusinessTravel ===
   BusinessTravel  total  attrition_pct
Travel_Frequently    277           24.9
    Travel_Rarely   1043           15.0
       Non-Travel    150            8.0


,BusinessTravel,total,attrition_pct
0,Travel_Frequently,277,24.9
1,Travel_Rarely,1043,15.0
2,Non-Travel,150,8.0


**All Phase 1 categorical findings confirmed with SQL:**

- Q5: Sales (20.6%) > HR (19.0%) > R&D (13.8%)
- Q6: Sales Rep (39.8%) highest, Manager (5%) lowest
- Q7: Single (25.5%) > Married (12.5%) > Divorced (10.1%)
- Q8: Frequent travelers (24.9%) > Rare (15%) > Non-travel (8%)


## Step 4 -- HAVING: Filter After Grouping

**Most common SQL interview question:** What is the difference between WHERE and HAVING?

- `WHERE` filters rows BEFORE grouping
- `HAVING` filters groups AFTER grouping (when condition uses COUNT or AVG)


In [49]:
#### 5. High-attrition roles with 50+ employees (HAVING)
sql('''
    SELECT "JobRole", COUNT(*) AS total,
           ROUND((AVG(CAST("Attrition" AS FLOAT)) * 100.0)::numeric, 1) AS attrition_pct
    FROM employees GROUP BY "JobRole"
    HAVING AVG(CAST("Attrition" AS FLOAT)) > 0.20 AND COUNT(*) >= 50
    ORDER BY attrition_pct DESC
''', 'Q9: High-attrition roles with 50+ employees (HAVING)')


=== Q9: High-attrition roles with 50+ employees (HAVING) ===
              JobRole  total  attrition_pct
 Sales Representative     83           39.8
Laboratory Technician    259           23.9
      Human Resources     52           23.1


,JobRole,total,attrition_pct
0,Sales Representative,83,39.8
1,Laboratory Technician,259,23.9
2,Human Resources,52,23.1


In [50]:
#### 10. Age groups with 100+ employees (HAVING + CASE WHEN)
sql('''
    SELECT
        CASE WHEN "Age" < 25 THEN 'Under 25'
             WHEN "Age" <= 34 THEN '25-34'
             WHEN "Age" <= 44 THEN '35-44'
             ELSE '45+' END AS age_group,
        COUNT(*) AS total,
        ROUND((AVG(CAST("Attrition" AS FLOAT)) * 100.0)::numeric, 1) AS attrition_pct
    FROM employees GROUP BY age_group
    HAVING COUNT(*) > 100 ORDER BY attrition_pct DESC
''', 'Q10: Age groups with 100+ employees (HAVING + CASE WHEN)')



=== Q10: Age groups with 100+ employees (HAVING + CASE WHEN) ===
age_group  total  attrition_pct
    25-34    554           20.2
      45+    314           11.5
    35-44    505           10.1


,age_group,total,attrition_pct
0,25-34,554,20.2
1,45+,314,11.5
2,35-44,505,10.1


In [52]:
#### 11. Departments below company average income (HAVING + Subquery)
sql('''
    SELECT "Department", ROUND((AVG("MonthlyIncome"))::Numeric,2) AS avg_income, COUNT(*) AS employees
    FROM employees GROUP BY "Department"
    HAVING AVG("MonthlyIncome") < (SELECT AVG("MonthlyIncome") FROM employees)
    ORDER BY avg_income ASC
''', 'Q11: Departments below company average income')


=== Q11: Departments below company average income ===
            Department  avg_income  employees
Research & Development     6281.25        961


,Department,avg_income,employees
0,Research & Development,6281.25,961


**WHERE vs HAVING:**

```sql
WHERE MonthlyIncome > 5000       -- filter rows BEFORE grouping
HAVING AVG(MonthlyIncome) > 5000 -- filter groups AFTER grouping
```


## Step 5 -- CASE WHEN: Conditional Logic

**Creates new categories in SQL** -- like `pd.cut()` in Python. These queries recreate Phase 1 risk segmentation in SQL.


In [54]:
#### 12. Risk segments (CASE WHEN)
sql('''
    SELECT
        CASE WHEN "OverTime" = 1 AND "MonthlyIncome" < 5000 THEN \'Extreme Risk\'
             WHEN "OverTime" = 1 THEN \'High Risk OT\'
             WHEN "MonthlyIncome" < 3000 THEN \'High Risk Pay\'
             ELSE \'Normal\' END AS risk_segment,
        COUNT(*) AS employees,
        ROUND((AVG(CAST("Attrition" AS FLOAT)) * 100.0)::numeric, 1) AS attrition_pct
    FROM employees GROUP BY risk_segment ORDER BY attrition_pct DESC
''', 'Q12: Risk segments (CASE WHEN)')



=== Q12: Risk segments (CASE WHEN) ===
 risk_segment  employees  attrition_pct
 Extreme Risk        205           42.9
 High Risk OT        211           18.5
High Risk Pay        281           17.4
       Normal        773            7.9


,risk_segment,employees,attrition_pct
0,Extreme Risk,205,42.9
1,High Risk OT,211,18.5
2,High Risk Pay,281,17.4
3,Normal,773,7.9


In [56]:
#### 13. Attrition by income band (CASE WHEN)
sql('''
    SELECT
        CASE WHEN "MonthlyIncome" < 3000  THEN 'Low  (<3k)'
             WHEN "MonthlyIncome" < 6000  THEN 'Mid  (3k-6k)'
             WHEN "MonthlyIncome" < 10000 THEN 'High (6k-10k)'
             ELSE 'Very High (>10k)' END AS income_band,
        COUNT(*) AS employees,  
        ROUND((AVG(CAST("Attrition" AS FLOAT)) * 100.0)::numeric, 1) AS attrition_pct
    FROM employees GROUP BY income_band ORDER BY attrition_pct DESC
''', 'Q13: Attrition by income band')


=== Q13: Attrition by income band ===
     income_band  employees  attrition_pct
      Low  (<3k)        395           28.6
    Mid  (3k-6k)        519           12.7
   High (6k-10k)        275           12.0
Very High (>10k)        281            8.9


,income_band,employees,attrition_pct
0,Low (<3k),395,28.6
1,Mid (3k-6k),519,12.7
2,High (6k-10k),275,12.0
3,Very High (>10k),281,8.9


**Q12 confirms Phase 1 Finding 3:** Extreme Risk segment (Overtime + low income) shows highest attrition.


## Step 6 -- Subqueries

**A SELECT nested inside another SELECT.** The inner query runs first. Equivalent to Phase 1's `groupby().transform('mean')`.


In [59]:
#### 14. Below-dept-average earners attrition (Subquery in WHERE)
sql('''
    SELECT e."Department", COUNT(*) AS below_avg_count,
           ROUND((AVG(CAST(e."Attrition" AS FLOAT)) * 100.0)::numeric, 1) AS attrition_pct
    FROM employees e
    WHERE e."MonthlyIncome" < (SELECT AVG("MonthlyIncome") FROM employees sub WHERE sub."Department" = e."Department")
    GROUP BY e."Department" ORDER BY attrition_pct DESC
''', 'Q14: Below-dept-average earners attrition')


=== Q14: Below-dept-average earners attrition ===
            Department  below_avg_count  attrition_pct
                 Sales              286           22.4
       Human Resources               46           21.7
Research & Development              655           17.6


,Department,below_avg_count,attrition_pct
0,Sales,286,22.4
1,Human Resources,46,21.7
2,Research & Development,655,17.6


In [61]:
#### 15. Above-company-average earners (Subquery in WHERE)
sql('''
    SELECT COUNT(*) AS above_avg_count,
           ROUND((AVG(CAST("Attrition" AS FLOAT)) * 100.0)::numeric, 1) AS attrition_pct
    FROM employees WHERE "MonthlyIncome" > (SELECT AVG("MonthlyIncome") FROM employees)
''', 'Q15: Above-company-average earners')


=== Q15: Above-company-average earners ===
 above_avg_count  attrition_pct
             493           10.5


,above_avg_count,attrition_pct
0,493,10.5


**Q14 confirms Phase 1 Finding 7:** Below-dept-average earners: 19.1% attrition vs 9.9% for above-average. Relative pay matters more than absolute pay.


## Step 7 -- Window Functions (High Interview Value)

**Compute values across rows WITHOUT collapsing them.** Unlike GROUP BY, every original row is kept.


In [62]:
#### 16. Income rank within each department (RANK)
sql('''
    SELECT "EmployeeNumber", "JobRole", "Department", "MonthlyIncome", "Attrition",
           RANK() OVER (PARTITION BY "Department" ORDER BY "MonthlyIncome" DESC) AS income_rank_in_dept
    FROM employees ORDER BY "Department", income_rank_in_dept LIMIT 12
''', 'Q16: Income rank within each department (RANK)')


=== Q16: Income rank within each department (RANK) ===
 EmployeeNumber         JobRole      Department  MonthlyIncome  Attrition  income_rank_in_dept
           1338         Manager Human Resources          19717          0                    1
           1625         Manager Human Resources          19658          0                    2
           1973         Manager Human Resources          19636          0                    3
            734         Manager Human Resources          19189          0                    4
            731         Manager Human Resources          19141          0                    5
            140         Manager Human Resources          18844          0                    6
            644         Manager Human Resources          18200          0                    7
            148         Manager Human Resources          17328          0                    8
           1408         Manager Human Resources          16799          0                

,EmployeeNumber,JobRole,Department,MonthlyIncome,Attrition,income_rank_in_dept
0,1338,Manager,Human Resources,19717,0,1
1,1625,Manager,Human Resources,19658,0,2
2,1973,Manager,Human Resources,19636,0,3
3,734,Manager,Human Resources,19189,0,4
4,731,Manager,Human Resources,19141,0,5
5,140,Manager,Human Resources,18844,0,6
6,644,Manager,Human Resources,18200,0,7
7,148,Manager,Human Resources,17328,0,8
8,1408,Manager,Human Resources,16799,0,9
9,1550,Manager,Human Resources,16437,0,10


In [63]:
#### 17. Running total by tenure (SUM OVER)
sql('''
    SELECT "YearsAtCompany", COUNT(*) AS employees,
           SUM(COUNT(*)) OVER (ORDER BY "YearsAtCompany") AS running_total,
           ROUND((AVG(CAST("Attrition" AS FLOAT)) * 100.0)::numeric, 1) AS attrition_pct
    FROM employees GROUP BY "YearsAtCompany" ORDER BY "YearsAtCompany" LIMIT 10
''', 'Q17: Running total by tenure (SUM OVER)')


=== Q17: Running total by tenure (SUM OVER) ===
 YearsAtCompany  employees  running_total  attrition_pct
              0         44           44.0           36.4
              1        171          215.0           34.5
              2        127          342.0           21.3
              3        128          470.0           15.6
              4        110          580.0           17.3
              5        196          776.0           10.7
              6         76          852.0           11.8
              7         90          942.0           12.2
              8         80         1022.0           11.3
              9         82         1104.0            9.8


,YearsAtCompany,employees,running_total,attrition_pct
0,0,44,44.0,36.4
1,1,171,215.0,34.5
2,2,127,342.0,21.3
3,3,128,470.0,15.6
4,4,110,580.0,17.3
5,5,196,776.0,10.7
6,6,76,852.0,11.8
7,7,90,942.0,12.2
8,8,80,1022.0,11.3
9,9,82,1104.0,9.8


In [64]:
#### 18. Lowest-paid employee per department (ROW_NUMBER)
sql('''
    SELECT "Department", "JobRole", "MonthlyIncome", "Attrition"
    FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY "Department" ORDER BY "MonthlyIncome" ASC) AS rn FROM employees) ranked
    WHERE rn = 1
''', 'Q18: Lowest-paid employee per department (ROW_NUMBER)')


=== Q18: Lowest-paid employee per department (ROW_NUMBER) ===
            Department              JobRole  MonthlyIncome  Attrition
       Human Resources      Human Resources           1555          1
Research & Development   Research Scientist           1009          1
                 Sales Sales Representative           1052          0


,Department,JobRole,MonthlyIncome,Attrition
0,Human Resources,Human Resources,1555,1
1,Research & Development,Research Scientist,1009,1
2,Sales,Sales Representative,1052,0


**Window function syntax:**

```sql
FUNCTION() OVER (
    PARTITION BY column  -- group rows, keep all rows
    ORDER BY column      -- sort within group
)
```

Q18 (ROW_NUMBER to pick the Nth row per group) is a classic interview pattern.


## Step 8 -- CTEs: Named Temporary Queries

**CTEs are like variables for SQL.** Syntax: `WITH name AS (SELECT ...)`. Use when a query has more than 2 steps -- more readable than nested subqueries.


In [ ]:
#### 19. High-risk employees by department (CTE)`
sql('''
    WITH high_risk AS (
        SELECT * FROM employees WHERE "OverTime" = 1 AND "JobSatisfaction" <= 2
    )
    SELECT "Department", COUNT(*) AS high_risk_count,
           ROUND((AVG(CAST("Attrition" AS FLOAT)) * 100.0)::numeric, 1) AS attrition_pct
    FROM high_risk GROUP BY "Department" ORDER BY attrition_pct DESC
''', 'Q19: High-risk employees by department (CTE)')


=== Q19: High-risk employees by department (CTE) ===
            Department  high_risk_count  attrition_pct
                 Sales               40           50.0
       Human Resources                9           44.4
Research & Development              104           30.8


,Department,high_risk_count,attrition_pct
0,Sales,40,50.0
1,Human Resources,9,44.4
2,Research & Development,104,30.8


In [66]:
#### 20. The danger zone -- all 4 risk factors (CTE)

sql('''
    WITH company_avg AS (
        SELECT AVG("MonthlyIncome") AS avg_income, AVG("YearsAtCompany") AS avg_tenure FROM employees
    ),
    danger_zone AS (
        SELECT e.* FROM employees e JOIN company_avg c ON 1=1
        WHERE e."OverTime" = 1 AND e."MonthlyIncome" < c.avg_income
          AND e."YearsAtCompany" < c.avg_tenure AND e."StockOptionLevel" = 0
    )
    SELECT COUNT(*) AS danger_zone_count,
           ROUND((AVG(CAST("Attrition" AS FLOAT)) * 100.0)::numeric, 1) AS attrition_pct
    FROM danger_zone
''', 'Q20: The danger zone -- all 4 risk factors (CTE)')


=== Q20: The danger zone -- all 4 risk factors (CTE) ===
 danger_zone_count  attrition_pct
                94           58.5


,danger_zone_count,attrition_pct
0,94,58.5


**Q19 confirms IsHighRisk from Phase 1.** Q20 confirms Phase 1 Finding 3 -- the danger zone produces extreme attrition.


## Step 9 -- Star Schema: CREATE TABLE

**Foundation of Phase 6 (Data Warehousing) and Phase 7 (Power BI).** Star schema = fact table in the middle + dimension tables around it.


In [3]:
schema_stmts = [
    'CREATE TABLE IF NOT EXISTS dim_department (dept_id INTEGER PRIMARY KEY, dept_name VARCHAR(100), budget_m DECIMAL(8,2))',
    'CREATE TABLE IF NOT EXISTS dim_job (job_id INTEGER PRIMARY KEY, job_role VARCHAR(100), job_level INTEGER)',
    'CREATE TABLE IF NOT EXISTS dim_employee (employee_id INTEGER PRIMARY KEY, age INTEGER, gender VARCHAR(10), marital_status VARCHAR(20), dept_id INTEGER, job_id INTEGER)',
    'CREATE TABLE IF NOT EXISTS fact_attrition (fact_id INTEGER PRIMARY KEY, employee_id INTEGER, monthly_income DECIMAL(10,2), years_at_company INTEGER, overtime INTEGER, stock_option_level INTEGER, job_satisfaction INTEGER, attrition INTEGER)'
]
print('Creating star schema:')
for s in schema_stmts:
    print('  ' + s[:70] + '...')
try:
    with engine.connect() as conn:
        for stmt in schema_stmts:
            conn.execute(text(stmt))
        conn.commit()
    print('\nTables created: dim_department, dim_job, dim_employee, fact_attrition')
except Exception as e:
    print('\nNote: ' + str(e) + ' -- run DDL in pgAdmin')
print('\nSchema explanation:')
print('  dim_* tables = dimension tables (who, what, where -- descriptive)')
print('  fact_attrition = fact table (the event we are measuring)')
print('  This star schema is the data model for Phase 7 Power BI.')

Creating star schema:
  CREATE TABLE IF NOT EXISTS dim_department (dept_id INTEGER PRIMARY KEY...
  CREATE TABLE IF NOT EXISTS dim_job (job_id INTEGER PRIMARY KEY, job_ro...
  CREATE TABLE IF NOT EXISTS dim_employee (employee_id INTEGER PRIMARY K...
  CREATE TABLE IF NOT EXISTS fact_attrition (fact_id INTEGER PRIMARY KEY...

Tables created: dim_department, dim_job, dim_employee, fact_attrition

Schema explanation:
  dim_* tables = dimension tables (who, what, where -- descriptive)
  fact_attrition = fact table (the event we are measuring)
  This star schema is the data model for Phase 7 Power BI.


## Step 10 -- View for Easy Access


In [5]:
view_sql = """CREATE OR REPLACE VIEW v_attrition_summary AS 
SELECT 
    "Department", 
    "JobRole", 
    COUNT(*) AS total_employees, 
    SUM(CAST("Attrition" AS INTEGER)) AS employees_left, 
    ROUND((AVG(CAST("Attrition" AS FLOAT)) * 100.0)::numeric, 1) AS attrition_pct, 
    ROUND(AVG("MonthlyIncome"), 0) AS avg_monthly_income 
FROM employees 
GROUP BY "Department", "JobRole" 
ORDER BY attrition_pct DESC;
"""
print('Creating view: v_attrition_summary')
try:
    with engine.connect() as conn:
        conn.execute(text(view_sql))
        conn.commit()
    result = pd.read_sql('SELECT * FROM v_attrition_summary LIMIT 8', engine)
    print('\nView output:')
    print(result.to_string(index=False))
except Exception as e:
    print('Note: ' + str(e))
print('\nIndex examples for performance:')
print('  CREATE INDEX idx_dept ON employees(Department);')
print('  CREATE INDEX idx_attrition ON employees("Attrition");')

Creating view: v_attrition_summary

View output:
            Department                   JobRole  total_employees  employees_left  attrition_pct  avg_monthly_income
                 Sales      Sales Representative               83              33           39.8              2626.0
Research & Development     Laboratory Technician              259              62           23.9              3237.0
       Human Resources           Human Resources               52              12           23.1              4236.0
                 Sales           Sales Executive              326              57           17.5              6924.0
Research & Development        Research Scientist              292              47           16.1              3240.0
Research & Development Healthcare Representative              131               9            6.9              7529.0
Research & Development    Manufacturing Director              145              10            6.9              7295.0
Research & Deve

**View** = saved SQL query. HR can run `SELECT * FROM v_attrition_summary` without knowing any SQL internals. **Index** = makes WHERE and GROUP BY queries faster on large tables.


## SQL Cheat Sheet (Interview Ready)


In [69]:
cheat = (
    'SQL CHEAT SHEET\n'
    '\nGROUP BY (most used in analytics)\n'
    '  SELECT col, COUNT(*), AVG(target)\n'
    '  FROM table GROUP BY col ORDER BY AVG(target) DESC\n'
    '\nWHERE vs HAVING\n'
    '  WHERE  = filter rows  BEFORE grouping\n'
    '  HAVING = filter groups AFTER grouping\n'
    '\nCASE WHEN\n'
    '  CASE WHEN cond THEN label ELSE default END AS col\n'
    '\nWINDOW FUNCTIONS\n'
    '  RANK()       OVER (PARTITION BY dept ORDER BY income DESC)\n'
    '  ROW_NUMBER() OVER (PARTITION BY dept ORDER BY income ASC)\n'
    '  SUM(col)     OVER (ORDER BY date)  <- running total\n'
    '\nCTE\n'
    '  WITH cte AS (SELECT ...) SELECT ... FROM cte\n'
    '\nTOP 5 INTERVIEW QUESTIONS\n'
    '  1. GROUP BY + COUNT/AVG  - almost every interview\n'
    '  2. HAVING vs WHERE       - 80% of interviews\n'
    '  3. RANK or ROW_NUMBER    - 60% of interviews\n'
    '  4. Subquery              - 40% of interviews\n'
    '  5. CTE                   - 30% of interviews'
)
print(cheat)

SQL CHEAT SHEET

GROUP BY (most used in analytics)
  SELECT col, COUNT(*), AVG(target)
  FROM table GROUP BY col ORDER BY AVG(target) DESC

WHERE vs HAVING
  WHERE  = filter rows  BEFORE grouping
  HAVING = filter groups AFTER grouping

CASE WHEN
  CASE WHEN cond THEN label ELSE default END AS col

WINDOW FUNCTIONS
  RANK()       OVER (PARTITION BY dept ORDER BY income DESC)
  ROW_NUMBER() OVER (PARTITION BY dept ORDER BY income ASC)
  SUM(col)     OVER (ORDER BY date)  <- running total

CTE
  WITH cte AS (SELECT ...) SELECT ... FROM cte

TOP 5 INTERVIEW QUESTIONS
  1. GROUP BY + COUNT/AVG  - almost every interview
  2. HAVING vs WHERE       - 80% of interviews
  3. RANK or ROW_NUMBER    - 60% of interviews
  4. Subquery              - 40% of interviews
  5. CTE                   - 30% of interviews


## Phase 5 -- Summary

**20 SQL queries covering:**
SELECT/WHERE/ORDER BY (Q1-Q4), GROUP BY + aggregates (Q5-Q8), HAVING (Q9-Q11), CASE WHEN (Q12-Q13), Subqueries (Q14-Q15), Window functions (Q16-Q18), CTEs (Q19-Q20), Star schema DDL (Step 9), Views (Step 10).

**Every query answers a Phase 1 business question with SQL instead of Python.** This proves you can work in any data environment.

**Next:**

- Phase 6: Data Warehousing -- load data into star schema
- Phase 7: Power BI -- 4-page dashboard
